# GlyphBench Return Violins

Violin-only inspection of per-episode GlyphBench return distributions.

In [ ]:
import json
from contextlib import suppress
from pathlib import Path

import pandas as pd

from glyphbench.plotting import plot_return_violins
from glyphbench.plotting.rollouts import load_rollouts

with suppress(NameError):
    get_ipython().run_line_magic("config", "InlineBackend.figure_format = 'retina'")

# Pick exactly one data source.
# 1. Prime-rl training output root containing run_default/rollouts/.
RUN_DIR = None

# 2. Raw verifiers eval results root containing results.jsonl files.
#    Example shape: <root>/evals/glyphbench--Qwen--Qwen3.5-4B/<run-id>/results.jsonl
EVAL_RESULTS_DIR = Path("../results/evals/glyphbench--Qwen--Qwen3.5-4B")

MODEL = "Qwen/Qwen3.5-4B"


def _suite(env_id):
    if not isinstance(env_id, str) or "/" not in env_id:
        return None
    return env_id.split("/", 1)[1].split("-", 1)[0]


def _info(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, dict) else {}
        except json.JSONDecodeError:
            return {}
    return {}


def _row_return(row):
    value = row.get("reward", row.get("episodic_return"))
    if value is None and isinstance(row.get("metrics"), dict):
        value = row["metrics"].get("episodic_return")
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _metadata_model(results_jsonl):
    metadata = results_jsonl.parent / "metadata.json"
    if not metadata.exists():
        return None
    try:
        payload = json.loads(metadata.read_text())
    except (OSError, json.JSONDecodeError):
        return None
    return payload.get("model") or payload.get("model_id") or payload.get("model_name")


def _load_eval_results(results_dir, model=None):
    rows = []
    for results_jsonl in Path(results_dir).rglob("results.jsonl"):
        run_model = _metadata_model(results_jsonl)
        if model is not None and run_model != model:
            continue
        with results_jsonl.open("r", encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if row.get("error") is not None or row.get("stop_condition") == "has_error":
                    continue
                if row.get("error_chain") or row.get("long_error_chain"):
                    continue
                info = _info(row.get("info"))
                env_id = info.get("env_id")
                value = _row_return(row)
                if env_id is None or value is None:
                    continue
                rows.append({
                    "phase": "eval",
                    "model": run_model,
                    "env_id": env_id,
                    "suite": _suite(env_id),
                    "episodic_return": value,
                })
    return pd.DataFrame(rows)


def load_violin_frame(
    run_dir=RUN_DIR,
    eval_results_dir=EVAL_RESULTS_DIR,
    model=MODEL,
):
    if run_dir is not None:
        run_dir = Path(run_dir)
        if (run_dir / "run_default" / "rollouts").exists():
            df = load_rollouts(run_dir, phases=("eval",))
            if not df.empty:
                return df

    if eval_results_dir is not None:
        df = _load_eval_results(eval_results_dir, model=model)
        if not df.empty:
            return df

    raise FileNotFoundError(
        "No data found. Set RUN_DIR to a prime-rl run root, or "
        "EVAL_RESULTS_DIR to a raw eval results root."
    )


df = load_violin_frame()

In [ ]:
# One figure only. Use "suite" for the compact overview or "env_id" for the wide per-task view.
GROUP_BY = "suite"  # "suite" or "env_id"
MAX_TASKS = 80       # only used when GROUP_BY == "env_id"
SAVE_PATH = None     # e.g. Path("return_violins.png")

plot_df = df.copy()
order = None
title = "GlyphBench suite return distributions"

if GROUP_BY == "env_id":
    order = (
        plot_df.groupby("env_id")["episodic_return"]
        .median()
        .sort_values(ascending=False)
        .index
        .tolist()
    )[:MAX_TASKS]
    plot_df = plot_df[plot_df["env_id"].isin(order)]
    title = f"GlyphBench task return distributions (top {len(order)} by median)"
elif GROUP_BY != "suite":
    raise ValueError('GROUP_BY must be "suite" or "env_id"')

fig, ax = plot_return_violins(
    plot_df,
    group_col=GROUP_BY,
    value_col="episodic_return",
    phase="eval",
    order=order,
    title=title,
    show_points=GROUP_BY == "suite",
)
fig.set_dpi(180)
if GROUP_BY == "env_id":
    fig.set_size_inches(max(10, 0.28 * len(order)), 4.4)
if SAVE_PATH is not None:
    fig.savefig(SAVE_PATH, dpi=300, facecolor=fig.get_facecolor())
fig